# Laboratorio 2 - Hashing y árbol Merkle

## Primera parte: creación del árbol de Merkle

Tenemos las condiciones:
- Cada hoja contiene el hash SHA-256 de un bloque de datos.
- Cada nodo interno contiene el hash SHA-256 de la concatenación de sus dos hijos.
- Si el número de hojas es impar, la última hoja se duplica.
- La raíz (Merkle Root) es el hash que representa todo el conjunto.

### ACLARACIÓN IMPORTANTE:

La implementación del siguiente árbol de Merkle es una adaptación de la implementación obtenida de Introduction to Merkle Tree de GeeksForGeeks.

Enlace: https://www.geeksforgeeks.org/dsa/introduction-to-merkle-tree/

In [1]:
import hashlib

In [ ]:
# Podemos reutilizar la clase ArbolMerkle que puede encontrarse en la carpeta **reto_1_hashing_merkle_tree**

"""
    Clase que se usa para representar un nodo del árbol.
    Se presupone que el árbol será binario, por lo que
    un nodo tendrá hijos izquierdo y derecho.

    La clase contiene como atributos:
    - Hijo izquierdo : Nodo
    - Hijo derecho : Nodo
    - valor_hash : str
    - contenido (la transacción que representa ese nodo) : str
    - esta_copiado : bool
"""
class Nodo:
    def __init__(self, izquierdo, derecho, valor_hash, contenido, esta_copiado = False):
        self.izquierdo = izquierdo
        self.derecho = derecho
        self.valor_hash = valor_hash
        self.contenido = contenido
        self.esta_copiado = esta_copiado

    @staticmethod
    def aplicar_hash(valor : str) -> str:
        return hashlib.sha256(valor.encode()).hexdigest()

    def generar_copia(self):
        return Nodo(self.izquierdo, self.derecho, self.valor_hash, self.contenido, True)

    def __str__(self):
        return "Nodo = " + str(self.valor_hash)

    
"""
    La siguiente clase representa un árbol de Merkle. Solo recibe la lista de transacciones.
    Implementa la estrategia de duplicar el nodo impar.
"""
class ArbolMerkle:

    def __init__(self, valores: list[str]):
        self.raiz = None
        self._crear_arbol(valores=valores)
        

    def _crear_arbol(self, valores:list[str]):
        """
            En este método se preparan los valores, que originalmente son cadenas,
            transformándolos en objetos de la clase Nodo, aquí se obtienen las hojas.

            Args:
            valores (list[str]) : es la lista de cadenas que serán las hojas.
        """
        hojas = []

        for valor in valores:
            nodo = Nodo(
                izquierdo= None,
                derecho= None,
                valor_hash= Nodo.aplicar_hash(valor),
                contenido= valor
            )

            hojas.append(nodo)

        if len(hojas) % 2 == 1:
            # Si la cantidad de hojas es impar, entonces duplicamos el último nodo
            ultimo_nodo = hojas[-1]
            hojas.append(ultimo_nodo.generar_copia())

        # Una vez obtenida la lista de nodos, se deben colocar en forma de árbol binario
        # Esto se logra recursivamente
        self.raiz = self._crear_arbol_recursivamente(hojas)

    def _crear_arbol_recursivamente(self, lista_nodos):


        # Si hay cantidad impar de nodos, copiamos el último de la lista
        # Así aseguramos siempre que un nodo padre tenga dos hijos
        if len(lista_nodos) % 2 != 0:
            ultimo_nodo = lista_nodos[-1]
            nodo_duplicado = ultimo_nodo.generar_copia()

            lista_nodos.append(nodo_duplicado)


        # Caso base: Si la lista tiene solo dos nodos, los convertimos en
        # hijos de un padre
        if len(lista_nodos) == 2:
            nodo_izquierdo = lista_nodos[0]
            nodo_derecho = lista_nodos[1]

            hash_padre = Nodo.aplicar_hash(nodo_izquierdo.valor_hash + nodo_derecho.valor_hash)

            contenido_padre = nodo_izquierdo.contenido + "+" + nodo_derecho.contenido

            return Nodo(
                izquierdo= nodo_izquierdo,
                derecho= nodo_derecho,
                valor_hash= hash_padre,
                contenido= contenido_padre
            )
        else:
            # Caso recursivo: construir el árbol por mitades

            mitad = len(lista_nodos) // 2 # Dividir lista en dos

            # Obtener dos sublistas
            nodos_izquierda = lista_nodos[:mitad] 
            nodos_derecha = lista_nodos[mitad:]


            # Subárbol izquierdo
            hijo_izquierdo = self._crear_arbol_recursivamente(nodos_izquierda)

            # Subárbol derecho
            hijo_derecho = self._crear_arbol_recursivamente(nodos_derecha)


            # Creamos el nodo padre que conecta los subárboles
            hash_padre = Nodo.aplicar_hash(hijo_izquierdo.valor_hash + hijo_derecho.valor_hash)

            contenido_padre = hijo_izquierdo.contenido + "+" + hijo_derecho.contenido

            # Devolvemos el nodo padre

            return Nodo(
                izquierdo=hijo_izquierdo,
                derecho=hijo_derecho,
                valor_hash=hash_padre,
                contenido=contenido_padre
            )

    def get_hash_raiz(self):
        return self.raiz.valor_hash

    def mostrar_arbol(self):
        """
            Este método se encarga de imprimir el árbol en pantalla
        """
        self._mostrar_arbol_recursivamente(self.raiz)

    def _mostrar_arbol_recursivamente(self, nodo:Nodo):

        # Caso base: si el nodo no existe, no muestra nada
        if nodo is None:
            return

        # Mostrar el valor hash y su contenido, vamos a truncar el hash a solo 4 términos
        print("Valor Hash de este Nodo: ", nodo.valor_hash[:4], "...")
        print("Contenido del nodo: ", nodo.contenido)
        

        # Validar si el nodo es una hoja
        if nodo.izquierdo is None and nodo.derecho is None:
            print("Hoja (Input)")
            if nodo.esta_copiado:
                print("(Duplicado)")
            print("\n")
        else:
            print("Hijo Izquierdo: ", nodo.izquierdo)
            print("Hijo Derecho: ", nodo.derecho)
            if nodo.esta_copiado:
                print("(Duplicado)")
            print("\n")

        # Recorrer recursivamente el subárbol izquierdo
        self._mostrar_arbol_recursivamente(nodo.izquierdo)
        # Recorrer recursivamente el subárbol derecho
        self._mostrar_arbol_recursivamente(nodo.derecho)

    def obtener_prueba_inclusion(self, dato):
        """
            Genera la lista de los hashes hermanos al recorrer desde la hoja del dato hasta la ráiz, necesario para la verificación de la prueba de inclusión
        """

        prueba = []

        encontrado = self._buscar_dato_recursivamente(
                self.raiz,
                dato,
                prueba
            )

        if encontrado:
            return prueba

        return None

    def _buscar_dato_recursivamente(self, nodo: Nodo, dato, prueba):

        # No existe este nodo
        if nodo is None:
            return False

        # Validar si llegamos a una hoja
        if nodo.izquierdo is None and nodo.derecho is None:

            if nodo.contenido == dato:
                return True

            return False

        # Primero buscamos por la izquierda
        encontrado = self._buscar_dato_recursivamente(
            nodo.izquierdo,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la izquierda.
            # Guardamos el hash del hermano derecho.
            prueba.append(
                (nodo.derecho.valor_hash, "derecha")
            )

            return True

        # Si no estaba a la izquierda,
        # buscamos por la derecha
        encontrado = self._buscar_dato_recursivamente(
            nodo.derecho,
            dato,
            prueba
        )

        if encontrado:

            # El dato estaba a la derecha.
            # Guardamos el hash del hermano izquierdo.
            prueba.append(
                (nodo.izquierdo.valor_hash, "izquierda")
            )

            return True

        return False

            


In [13]:
## EXPERIMENTOS
bloques_datos = ["T1 : Buen día", "T2 : Buena noche", "T3 : Hola", "T4 : Adios", "T5 : Hasta Pronto"]


## Creamos el árbol:
print("#################### ÁRBOL ####################")

arbol = ArbolMerkle(bloques_datos)
print("---> RAÍZ: ", arbol.get_hash_raiz())
arbol.mostrar_arbol()

#################### ÁRBOL ####################
---> RAÍZ:  606963a4bb7ef5949b2b89b6db553fc6a9ea2291d1e37bbcb2eeeaf6944ca1d0
Valor Hash de este Nodo:  6069 ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche+T3 : Hola+T3 : Hola+T4 : Adios+T5 : Hasta Pronto+T5 : Hasta Pronto+T5 : Hasta Pronto
Hijo Izquierdo:  Nodo = 3bb9e8d2a5983d2dcf7608478bdf36f4ba397abe656501c07a124f557045e335
Hijo Derecho:  Nodo = 1fdc8e533760076cbc865ee370be1104b29b99ac043998c994b43c37e4186fe7


Valor Hash de este Nodo:  3bb9 ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche+T3 : Hola+T3 : Hola
Hijo Izquierdo:  Nodo = c616d02a4b793b958979237d07b153358ab4143d245be2584ef7b6a9e1b81efd
Hijo Derecho:  Nodo = b154d92ddaffc8c3172d27a9cb83e70fea58bab6ab5fd8b11bcb3fbbe63a0e3c


Valor Hash de este Nodo:  c616 ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche
Hijo Izquierdo:  Nodo = 2c5c29b6544929b97681f4c199658e62360de2b73bcb4a2a84c23075813549e8
Hijo Derecho:  Nodo = 7ae2eb91c3def4aa8cac1a8f9c6004da7b06f21170

In [14]:
# EXPERIMENTO 2: Modificar una transacción
nuevos_bloques = ["T1 : Buen día", "T2 : Buena noche", "T3 : Hola", "T4 : Adios", "T77889900"]

## Creamos el árbol:
print("#################### ÁRBOL (NUEVO) ####################")
arbol2 = ArbolMerkle(nuevos_bloques)
print("---> RAÍZ: ", arbol2.get_hash_raiz())
arbol2.mostrar_arbol()

#################### ÁRBOL (NUEVO) ####################
---> RAÍZ:  72ad107b765bdf5e4c99292b1d3a33cbe3b6d6de1b380c4cd3983446b03f7518
Valor Hash de este Nodo:  72ad ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche+T3 : Hola+T3 : Hola+T4 : Adios+T77889900+T77889900+T77889900
Hijo Izquierdo:  Nodo = 3bb9e8d2a5983d2dcf7608478bdf36f4ba397abe656501c07a124f557045e335
Hijo Derecho:  Nodo = ae221376154006225d410fafd08ca6000c805bb969fd926b3b66702d4ebf67a7


Valor Hash de este Nodo:  3bb9 ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche+T3 : Hola+T3 : Hola
Hijo Izquierdo:  Nodo = c616d02a4b793b958979237d07b153358ab4143d245be2584ef7b6a9e1b81efd
Hijo Derecho:  Nodo = b154d92ddaffc8c3172d27a9cb83e70fea58bab6ab5fd8b11bcb3fbbe63a0e3c


Valor Hash de este Nodo:  c616 ...
Contenido del nodo:  T1 : Buen día+T2 : Buena noche
Hijo Izquierdo:  Nodo = 2c5c29b6544929b97681f4c199658e62360de2b73bcb4a2a84c23075813549e8
Hijo Derecho:  Nodo = 7ae2eb91c3def4aa8cac1a8f9c6004da7b06f211705ce2374f32f74f25

In [12]:
# Comprobamos las dos raíces
if arbol.get_hash_raiz() == arbol2.get_hash_raiz():
    print("Las Raíces son IGUALES.")
else:
    print("Las Raíces son DIFERENTES.")
    

Las Raíces son DIFERENTES.


## Segunda parte: prueba de inclusión

Para este apartado, aprovechamos el método que genera la prueba y utilizamos la función que la verifica.

In [15]:
def verificar_prueba_inclusion(dato, prueba, raiz):
    """
        Esta función se encarga de verificar la prueba generada por el método obtener_prueba_inclusion de
        la clase ArbolMerkle

        Aplicamos el algoritmo de clase:
        1. Hallamos hash del dato
        2. Hallamos hash de la concatenación del hash del dato y el de su hermano.
        3. Obtenemos al padre.
        4. Realizamos el mismo procedimiento entre el padre y su propio hermano hasta llegar de regreso a la raíz
        5. Comparamos si el hash de la raíz y el hash calculado coinciden.
    """

    # Primero calculamos hash del dato (la hoja)
    hash_actual = Nodo.aplicar_hash(dato)

    # Recorremos los hashes hermanos de cada nivel
    for hash_hermano, posicion in prueba:

        if posicion == "izquierda":

            hash_actual = Nodo.aplicar_hash(
                hash_hermano + hash_actual
            )

        else:

            hash_actual = Nodo.aplicar_hash(
                hash_actual + hash_hermano
            )

    # Verificar si llegamos a la misma raíz
    return hash_actual == raiz

In [17]:
# Prueba de inclusión para el bloque 3

bloques_datos = ["T1 : Buen día", "T2 : Buena noche", "T3 : Hola", "T4 : Adios", "T5 : Hasta Pronto"]


## Creamos el árbol sin mostrarlo (es el mismo de la sección anterior)

arbol = ArbolMerkle(bloques_datos)

transaccion3 = "T3 : Hola"
prueba = arbol.obtener_prueba_inclusion(dato=transaccion3)
if prueba is not None:
    resultado = verificar_prueba_inclusion(dato=transaccion3, prueba=prueba, raiz=arbol.get_hash_raiz())
    if resultado:
        print(f"Fue encontrada la transacción {transaccion3}")
    else:
        print("Transacción no encontrada.")

Fue encontrada la transacción T3 : Hola


In [21]:
# Intentamos con un dato incorrecto
bloques_datos = ["T1 : Buen día", 
                 "T2 : Buena noche", 
                 "T3 : Hola como estas", 
                 "T4 : Adios", 
                 "T5 : Hasta Pronto"]


## Creamos el árbol sin mostrarlo (es el mismo de la sección anterior)

arbol = ArbolMerkle(bloques_datos)

transaccion3 = "T3 : Hola como estas"
prueba = arbol.obtener_prueba_inclusion(dato=transaccion3)

transaccion_diferente = "T3"
if prueba is not None:
    resultado = verificar_prueba_inclusion(dato=transaccion_diferente, prueba=prueba, raiz=arbol.get_hash_raiz())
    if resultado:
        print(f"Fue encontrada la transacción {transaccion_diferente}")
    else:
        print(f"Transacción no encontrada en el árbol.")

Transacción no encontrada en el árbol.
